# 05. Hyperparameter Tuning — ResNet-18 Fully Fine-Tuned (Approach 3)

**Project:** Road Sign Recognition Using Deep Learning (Cambodian Traffic Signs, CamTSD), 26 classes

### Baseline results (already trained, not retrained here)

| Approach | Test accuracy | Macro F1 | Trainable parameters |
|---|---:|---:|---:|
| 1. SmallCNN from scratch | 70.19% | 50.78% | 133,178 |
| 2. ResNet-18 frozen backbone | 92.44% | 88.37% | 13,338 |
| **3. ResNet-18 fully fine-tuned** | **98.06%** | **96.80%** | 11,189,850 |

Approach 3 is the best baseline, so it is the model we tune.

### What is tuned

A simple, explicit 2 × 2 grid. Only **learning rate** and **weight decay** change; everything else stays exactly as in Approach 3.

| Config | Learning rate | Weight decay | Note |
|---|---:|---:|---|
| A | 1e-3 | 1e-4 | same settings as the Approach 3 baseline |
| B | 1e-3 | 1e-3 | stronger regularisation |
| C | 1e-4 | 1e-4 | smaller steps |
| D | 1e-4 | 1e-3 | smaller steps + stronger regularisation |

Fixed for every configuration: the same split (2,153 / 454 / 463), ImageNet `IMAGENET1K_V1` weights, all layers trainable, 224×224 inputs, the same train and val/test transforms, Adam, batch size 32, max 40 epochs, `ReduceLROnPlateau`, early stopping patience 10, seed 42.

### Why tune the learning rate?
The learning rate sets how big each weight update is. Fine-tuning starts from ImageNet weights that are already useful. A large step (1e-3) adapts quickly but can overwrite those features, while a small step (1e-4) adjusts them gently but may need more epochs. It is usually the single most influential hyperparameter.

### Why tune weight decay?
Weight decay is our **regularisation** choice. It penalises large weights, pushing the model towards simpler solutions that generalise better. With only 2,153 training images and 11.2M trainable parameters, the model can overfit. Stronger weight decay (1e-3) may reduce overfitting, but too much can cause underfitting.

### Why choose with the validation set, not the test set?
Hyperparameters are part of the model design. If we picked them by looking at test accuracy, the test set would effectively become training data for our decisions, and the reported test score would be optimistically biased. The validation set exists for exactly this: comparing models and choosing between them.

### Why must the test set stay untouched during tuning?
The test set is our only unbiased estimate of how the final model performs on new road signs. It is used **once**, after the configuration has been chosen on validation accuracy alone. In this notebook the test loader is not even created until Section 8.

### Protocol
1. Train configurations A–D on **train**, select each run's best epoch on **validation** (`src/training/train.py`).
2. Compare the four runs on **validation accuracy only**, and select the best one.
3. Reload the selected run's best-validation checkpoint (it already used the full Approach 3 protocol, so no retraining is needed).
4. Evaluate it on the **test** set exactly once, and compare with the 98.06% baseline.

**Runtime:** Google Colab with a GPU. Four full fine-tuning runs take roughly 4× the Approach 3 training time. If Colab disconnects, re-run the notebook: finished configurations are reloaded from Drive instead of retrained (`RESUME = True`).

## 1. Setup

### 1.1 Environment, Google Drive and repository

**Expected Google Drive layout** (edit `DRIVE_PROJECT_DIR` if yours differs):

```
MyDrive/Road_Sign_Recognition_CNN/
└── dataset/
    └── processed/                 # crops produced by notebook 01
        ├── train/<CLASS_NAME>/*.jpg
        ├── val/<CLASS_NAME>/*.jpg
        └── test/<CLASS_NAME>/*.jpg
```
A `dataset/processed.zip` whose top-level folder is `processed/` also works.

The code and the portable split manifests (`dataset/splits/*.csv`, with crop paths relative to `dataset/processed/`) come from the GitHub repository and are used as-is. Results and checkpoints are written to Drive, outside Git.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# ---- Edit these if your setup differs ------------------------------------
REPO_URL = "https://github.com/RothRothRoth/DeepLearning_CNN.git"
REPO_BRANCH = "main"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Road_Sign_Recognition_CNN")
# --------------------------------------------------------------------------

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/DeepLearning_CNN")
    if (PROJECT_ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True
        )

    SOURCE_DATASET_DIR = DRIVE_PROJECT_DIR / "dataset"         # processed crops on Drive (not in Git)
    PROCESSED_DIR = Path("/content/camtsd_dataset/processed")  # fast local copy used for training
    OUTPUT_ROOT = DRIVE_PROJECT_DIR / "results" / "runs" / "hyperparameter_tuning"
else:
    # Local execution (e.g. checking the notebook on a workstation).
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    SOURCE_DATASET_DIR = PROJECT_ROOT / "dataset"
    PROCESSED_DIR = SOURCE_DATASET_DIR / "processed"
    OUTPUT_ROOT = PROJECT_ROOT / "results" / "runs" / "hyperparameter_tuning"  # gitignored

# Optional override for where run outputs go.
OUTPUT_ROOT = Path(os.environ.get("TUNING_OUTPUT_ROOT", OUTPUT_ROOT))

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip() or "unknown"

print(f"Running in Colab:   {IN_COLAB}")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Git commit:         {GIT_COMMIT}")
print(f"Source dataset dir: {SOURCE_DATASET_DIR}")
print(f"Output root:        {OUTPUT_ROOT}")

### 1.2 Requirements

In [ ]:
import importlib

requirements = PROJECT_ROOT / "requirements.txt"
if requirements.exists() and requirements.read_text().strip():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
else:
    print("requirements.txt is empty; using the packages preinstalled in Colab.")

for pkg in ["torch", "torchvision", "numpy", "pandas", "PIL", "sklearn", "matplotlib"]:
    module = importlib.import_module(pkg)
    print(f"  {pkg:12s} {getattr(module, '__version__', '')}")

### 1.3 Project imports, PyTorch and GPU

In [ ]:
import dataclasses
import json
import platform
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torchvision.models import resnet18
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

from src.config import BATCH_SIZE, CLASS_NAMES, DEFAULT_SEED, IMAGE_SIZE, NUM_CLASSES, get_paths
from src.seed import set_seed
from src.data.dataset import get_dataloaders, resolve_crop_path

try:
    from src.models.resnet import (
        RESNET18_WEIGHTS,
        count_parameters,
        create_resnet18_finetuned,
    )
    from src.training import get_device, load_checkpoint, train_model, validate
except ImportError as exc:
    raise ImportError(
        "ResNet-18 / training modules not found. The cloned repository is older than "
        "commit 41ba0e4: push the local commits to GitHub, then re-run the setup cell."
    ) from exc

assert NUM_CLASSES == 26 and len(CLASS_NAMES) == 26, "Expected exactly 26 classes"

device = get_device()
print(f"PyTorch:      {torch.__version__}")
print(f"torchvision:  {torchvision.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} (CUDA {torch.version.cuda}, cuDNN {torch.backends.cudnn.version()})")
print(f"Device:       {device}")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(device)
    print(f"GPU:          {props.name}")
    print(f"GPU memory:   {props.total_memory / 1024**3:.1f} GB")
elif IN_COLAB:
    print("WARNING: no GPU detected. Use Runtime → Change runtime type → GPU.")

HARDWARE = {
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(device).total_memory / 1024**3, 2)
    if device.type == "cuda"
    else None,
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "colab": IN_COLAB,
}

### 1.4 Tuning configuration

`FIXED` holds everything shared by all four runs. These are the Approach 3 baseline settings. `CONFIGS` holds the only two values that change.

All outputs go to one fixed folder, so an interrupted session can resume:
`results/runs/hyperparameter_tuning/resnet18_finetuned_lr_wd_seed42/`

In [ ]:
SEED = DEFAULT_SEED  # 42

# Shared by ALL configurations: identical to the Approach 3 baseline.
FIXED = {
    "model_name": "resnet18_finetuned",
    "approach": "3_resnet18_fully_finetuned",
    "pretrained_weights": f"ResNet18_Weights.{RESNET18_WEIGHTS.name}",
    "trainable_layers": "all layers (full backbone + new Linear 512 -> 26)",
    "num_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "optimizer": "Adam (all parameters)",
    "loss": "CrossEntropyLoss",
    "max_epochs": 40,
    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "max",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "monitor": "val_acc",
    "early_stopping_patience": 10,
    "seed": SEED,
}

# The ONLY values that change: a 2 x 2 grid of learning rate x weight decay.
CONFIGS = [
    {"name": "A", "learning_rate": 1e-3, "weight_decay": 1e-4},  # = Approach 3 baseline settings
    {"name": "B", "learning_rate": 1e-3, "weight_decay": 1e-3},
    {"name": "C", "learning_rate": 1e-4, "weight_decay": 1e-4},
    {"name": "D", "learning_rate": 1e-4, "weight_decay": 1e-3},
]

# Selection rule, fixed BEFORE seeing any result (validation data only):
#   1. highest best validation accuracy
#   2. tie -> lowest validation loss at that best epoch
#   3. still tied -> earlier configuration in the list (A before B ...)
SELECTION_RULE = (
    "highest best validation accuracy; ties broken by lower validation loss at the best epoch, "
    "then by configuration order"
)

RESUME = True  # reuse configurations already finished in RUN_DIR (e.g. after a Colab disconnect)

RUN_DIR = OUTPUT_ROOT / f"resnet18_finetuned_lr_wd_seed{SEED}"
CHECKPOINT_DIR = RUN_DIR / "configs"          # one sub-folder per configuration
FIGURE_DIR = RUN_DIR / "figures"
METRICS_DIR = RUN_DIR / "metrics"
FINAL_DIR = RUN_DIR / "final_tuned_model"
for d in (CHECKPOINT_DIR, FIGURE_DIR, METRICS_DIR, FINAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"git_commit": GIT_COMMIT, "fixed": FIXED, "configs": CONFIGS,
               "selection_rule": SELECTION_RULE}, f, indent=2)

print(f"Run directory: {RUN_DIR}")
print("\nConfigurations to compare:")
print(pd.DataFrame(CONFIGS).rename(columns={"name": "Config", "learning_rate": "Learning rate",
                                             "weight_decay": "Weight decay"}).to_string(index=False))
print(f"\nSelection rule: {SELECTION_RULE}")

## 2. Dataset

### 2.1 Stage the processed crops

Reading thousands of small JPEGs directly from Drive is slow, so in Colab the `processed/` crops are copied once to the local runtime disk. Locally, the repository's `dataset/` folder is used directly.

In [ ]:
if IN_COLAB:
    drive_processed = SOURCE_DATASET_DIR / "processed"
    drive_zip = SOURCE_DATASET_DIR / "processed.zip"
    if PROCESSED_DIR.is_dir():
        print(f"Crops already staged at {PROCESSED_DIR}")
    elif drive_processed.is_dir():
        print(f"Copying {drive_processed} -> {PROCESSED_DIR} ...")
        t0 = time.perf_counter()
        shutil.copytree(drive_processed, PROCESSED_DIR)
        print(f"Done in {time.perf_counter() - t0:.0f}s")
    elif drive_zip.is_file():
        print(f"Extracting {drive_zip} -> {PROCESSED_DIR.parent} ...")
        shutil.unpack_archive(str(drive_zip), str(PROCESSED_DIR.parent))
    else:
        raise FileNotFoundError(
            f"No processed crops found. Expected {drive_processed}/ or {drive_zip} on Google Drive."
        )

if not PROCESSED_DIR.is_dir():
    raise FileNotFoundError(f"Processed crops not found at {PROCESSED_DIR}")
print("Splits present:", sorted(p.name for p in PROCESSED_DIR.iterdir() if p.is_dir()))

### 2.2 Load and verify the leak-free split manifests

The committed manifests in `dataset/splits/` are used directly. Each `crop_path` is a portable POSIX path relative to `dataset/processed/` (e.g. `train/KM_POST/<crop>.jpg`), resolved with `resolve_crop_path`, the same function `TrafficSignDataset` uses.

The only path override is `processed_dir`, which points at the local copy of the crops. This cell checks the split sizes, class coverage and zero parent-image overlap, and confirms that every crop file exists.

In [ ]:
EXPECTED_SIZES = {"train": 2153, "val": 454, "test": 463}

paths = dataclasses.replace(
    get_paths(str(PROJECT_ROOT)),     # manifests: <repo>/dataset/splits
    processed_dir=PROCESSED_DIR,      # crops: local copy (Colab) or <repo>/dataset/processed
    models_dir=CHECKPOINT_DIR,
    results_dir=RUN_DIR,
)
print(f"Manifests: {paths.splits_dir}")
print(f"Crops:     {paths.processed_dir}")

split_dfs = {s: pd.read_csv(paths.splits_dir / f"{s}.csv") for s in ("train", "val", "test")}

# Split sizes
for split, df in split_dfs.items():
    assert len(df) == EXPECTED_SIZES[split], f"{split}: {len(df)} rows, expected {EXPECTED_SIZES[split]}"
    assert (df["split"] == split).all(), f"{split}: rows with a different split label"

# Classes
for split, df in split_dfs.items():
    unknown = set(df["class_name"]) - set(CLASS_NAMES)
    assert not unknown, f"{split}: unknown classes {unknown}"
assert set(split_dfs["train"]["class_name"]) == set(CLASS_NAMES), "train split must cover all 26 classes"

# Zero parent-image overlap (no leakage)
parents = {s: set(df["parent_filename"]) for s, df in split_dfs.items()}
assert not parents["train"] & parents["val"], "train/val parent overlap"
assert not parents["train"] & parents["test"], "train/test parent overlap"
assert not parents["val"] & parents["test"], "val/test parent overlap"

# Every crop exists (resolve_crop_path also rejects non-portable paths)
missing = [
    p for df in split_dfs.values() for p in df["crop_path"]
    if not resolve_crop_path(paths.processed_dir, p).is_file()
]
assert not missing, f"{len(missing)} crop files missing, e.g. {missing[:3]}"

print("\nManifest checks passed.")
for split, df in split_dfs.items():
    print(f"  {split:5s}: {len(df):5d} crops from {len(parents[split]):5d} parent images")

class_dist = pd.DataFrame(
    {s: df["class_name"].value_counts() for s, df in split_dfs.items()}
).reindex(CLASS_NAMES).fillna(0).astype(int)
print("\nClass distribution:")
print(class_dist.to_string())

### 2.3 Train and validation DataLoaders (no test loader during tuning)

`make_train_val_loaders()` builds fresh, identically seeded loaders, so **every configuration sees exactly the same shuffling order and augmentation randomness**. The only difference between runs is then the hyperparameters.

The test loader returned by `get_dataloaders` is discarded here. The test set is loaded only in Section 8, after the configuration has been selected.

In [ ]:
def make_train_val_loaders():
    # Fresh train/val loaders with the same seed for every configuration (test loader discarded).
    set_seed(SEED)
    train_loader, val_loader, _unused_test_loader = get_dataloaders(
        paths=paths, batch_size=FIXED["batch_size"], num_workers=FIXED["num_workers"], seed=SEED
    )
    return train_loader, val_loader


train_loader, val_loader = make_train_val_loaders()
print(f"train: {len(train_loader.dataset):5d} samples, {len(train_loader):3d} batches")
print(f"val  : {len(val_loader.dataset):5d} samples, {len(val_loader):3d} batches")
print(f"test : {EXPECTED_SIZES['test']:5d} samples in the manifest, NOT loaded during tuning")
assert len(train_loader.dataset) == EXPECTED_SIZES["train"]
assert len(val_loader.dataset) == EXPECTED_SIZES["val"]

images, labels = next(iter(val_loader))
print(f"\nBatch images: {tuple(images.shape)}  labels: {tuple(labels.shape)}")
assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)

## 3. Model check (same architecture for every configuration)

Every configuration uses `create_resnet18_finetuned()`: ImageNet ResNet-18, new `Linear(512, 26)`, all layers trainable. This quick check runs once before tuning starts.

In [ ]:
set_seed(SEED)
check_model = create_resnet18_finetuned(num_classes=NUM_CLASSES).to(device)
TOTAL_PARAMS = count_parameters(check_model)
TRAINABLE_PARAMS = count_parameters(check_model, trainable_only=True)

assert all(p.requires_grad for p in check_model.parameters()), "some parameters are frozen"
assert TRAINABLE_PARAMS == TOTAL_PARAMS
assert (check_model.fc.in_features, check_model.fc.out_features) == (512, NUM_CLASSES)

with torch.no_grad():
    check_model.eval()
    check_out = check_model(images.to(device))
assert tuple(check_out.shape) == (images.shape[0], NUM_CLASSES)
assert torch.isfinite(check_out).all()

print(f"Classifier:           {check_model.fc}")
print(f"Total parameters:     {TOTAL_PARAMS:,}")
print(f"Trainable parameters: {TRAINABLE_PARAMS:,} (100% - full fine-tuning)")
print(f"Output shape:         {list(check_out.shape)}, finite: {torch.isfinite(check_out).all().item()}")

del check_model, check_out
if device.type == "cuda":
    torch.cuda.empty_cache()

## 4. Tuning: train configurations A–D

For each configuration the loop:
1. resets the seed and builds fresh train/val loaders;
2. builds a fresh pretrained ResNet-18, identical in every run;
3. trains with `train_model` from `src/training/train.py`, using the configuration's learning rate and weight decay;
4. keeps that run's **best-validation** checkpoint, and saves its history and a summary.

`train_model` receives only the train and validation loaders. The test set plays no part.

In [ ]:
criterion = nn.CrossEntropyLoss()
tuning_rows = []
tuning_histories = {}

for cfg in CONFIGS:
    name = cfg["name"]
    cfg_dir = CHECKPOINT_DIR / f"config_{name}"
    cfg_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = cfg_dir / "best_checkpoint.pt"
    history_path = cfg_dir / "history.json"
    summary_path = cfg_dir / "summary.json"

    print("=" * 72)
    print(f"Configuration {name}: learning rate = {cfg['learning_rate']:g}, weight decay = {cfg['weight_decay']:g}")
    print("=" * 72)

    if RESUME and ckpt_path.is_file() and history_path.is_file() and summary_path.is_file():
        print("Already finished in an earlier session -> loading saved results (RESUME = True)")
        with open(history_path) as f:
            history = json.load(f)
        with open(summary_path) as f:
            summary = json.load(f)
    else:
        train_loader, val_loader = make_train_val_loaders()
        set_seed(SEED)
        model = create_resnet18_finetuned(num_classes=NUM_CLASSES).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode=FIXED["scheduler_mode"],
            factor=FIXED["scheduler_factor"],
            patience=FIXED["scheduler_patience"],
        )
        hparams = {**FIXED, "config": name,
                   "learning_rate": cfg["learning_rate"], "weight_decay": cfg["weight_decay"]}

        history = train_model(
            model,
            train_loader,
            val_loader,
            optimizer,
            num_epochs=FIXED["max_epochs"],
            device=device,
            scheduler=scheduler,
            criterion=criterion,
            monitor=FIXED["monitor"],
            early_stopping_patience=FIXED["early_stopping_patience"],
            checkpoint_path=ckpt_path,
            model_name=f"resnet18_finetuned_config_{name}",
            num_classes=NUM_CLASSES,
            seed=SEED,
            hyperparameters=hparams,
        )

        best_i = history["epoch"].index(history["best_epoch"])
        summary = {
            "config": name,
            "learning_rate": cfg["learning_rate"],
            "weight_decay": cfg["weight_decay"],
            "best_val_acc": history["val_acc"][best_i],
            "val_loss_at_best_epoch": history["val_loss"][best_i],
            "best_val_loss": min(history["val_loss"]),
            "train_acc_at_best_epoch": history["train_acc"][best_i],
            "best_epoch": history["best_epoch"],
            "epochs_completed": history["epochs_completed"],
            "stopped_early": history["stopped_early"],
            "training_time_sec": history["total_time"],
            "trainable_parameters": count_parameters(model, trainable_only=True),
            "checkpoint": ckpt_path.relative_to(RUN_DIR).as_posix(),
        }
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)

        del model, optimizer, scheduler
        if device.type == "cuda":
            torch.cuda.empty_cache()

    tuning_rows.append(summary)
    tuning_histories[name] = history
    print(f"-> Config {name}: best val acc {summary['best_val_acc']:.4f} at epoch {summary['best_epoch']} "
          f"({summary['training_time_sec'] / 60:.1f} min, stopped early: {summary['stopped_early']})\n")

assert [r["config"] for r in tuning_rows] == [c["name"] for c in CONFIGS]
print("All configurations finished.")

## 5. Comparison table (validation only)

- **Best Val Accuracy** is the validation accuracy at the epoch kept as the checkpoint.
- **Best Val Loss** is the lowest validation loss seen during the run.
- **Val Loss @ Best Epoch** is the loss of the kept checkpoint, used to break ties.

No test numbers appear here.

In [ ]:
tuning_df = pd.DataFrame(tuning_rows)
table_df = pd.DataFrame({
    "Configuration": tuning_df["config"],
    "Learning Rate": tuning_df["learning_rate"],
    "Weight Decay": tuning_df["weight_decay"],
    "Best Val Accuracy": tuning_df["best_val_acc"],
    "Best Val Loss": tuning_df["best_val_loss"],
    "Val Loss @ Best Epoch": tuning_df["val_loss_at_best_epoch"],
    "Best Epoch": tuning_df["best_epoch"],
    "Epochs Run": tuning_df["epochs_completed"],
    "Early Stopped": tuning_df["stopped_early"],
    "Training Time (min)": tuning_df["training_time_sec"] / 60,
    "Trainable Parameters": tuning_df["trainable_parameters"],
})
table_df.to_csv(METRICS_DIR / "tuning_results.csv", index=False)

display_df = table_df.copy()
display_df["Learning Rate"] = display_df["Learning Rate"].map("{:g}".format)
display_df["Weight Decay"] = display_df["Weight Decay"].map("{:g}".format)
display_df["Best Val Accuracy"] = display_df["Best Val Accuracy"].map("{:.2%}".format)
for col in ("Best Val Loss", "Val Loss @ Best Epoch"):
    display_df[col] = display_df[col].map("{:.4f}".format)
display_df["Training Time (min)"] = display_df["Training Time (min)"].map("{:.1f}".format)
display_df["Trainable Parameters"] = display_df["Trainable Parameters"].map("{:,}".format)
print(display_df.to_string(index=False))
print(f"\nSaved {METRICS_DIR / 'tuning_results.csv'}")

## 6. Comparison figures

In [ ]:
names = [r["config"] for r in tuning_rows]
labels_cfg = [f"{r['config']}\nlr={r['learning_rate']:g}\nwd={r['weight_decay']:g}" for r in tuning_rows]
best_accs = [r["best_val_acc"] for r in tuning_rows]

# Figure 1: best validation accuracy per configuration
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels_cfg, best_accs, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
for bar, acc in zip(bars, best_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, acc, f"{acc:.2%}", ha="center", va="bottom", fontsize=10)
ax.set_ylim(max(0.0, min(best_accs) - 0.05), 1.005)
ax.set_ylabel("Best validation accuracy")
ax.set_title("Hyperparameter tuning (ResNet-18 fully fine-tuned): best validation accuracy")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "tuning_best_val_accuracy.png", dpi=150)
plt.show()
plt.close(fig)

# Figure 2: validation accuracy and loss per epoch for every configuration
fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(14, 5))
for r in tuning_rows:
    h = tuning_histories[r["config"]]
    label = f"{r['config']}: lr={r['learning_rate']:g}, wd={r['weight_decay']:g}"
    ax_acc.plot(h["epoch"], h["val_acc"], marker="o", markersize=3, label=label)
    ax_loss.plot(h["epoch"], h["val_loss"], marker="o", markersize=3, label=label)
ax_acc.set_title("Validation accuracy per epoch")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Validation accuracy")
ax_loss.set_title("Validation loss per epoch")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Validation loss")
for ax in (ax_acc, ax_loss):
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "tuning_val_curves.png", dpi=150)
plt.show()
plt.close(fig)
print(f"Saved figures to {FIGURE_DIR}")

## 7. Select the best configuration (validation accuracy only)

The rule was fixed in Section 1.4, before any run:
1. highest best validation accuracy;
2. if tied, lower validation loss at the best epoch, because a more confident model is preferred;
3. if still tied, earlier configuration in the list.

In [ ]:
ranked = sorted(
    tuning_rows,
    key=lambda r: (-r["best_val_acc"], r["val_loss_at_best_epoch"], [c["name"] for c in CONFIGS].index(r["config"])),
)
best = ranked[0]
tied_on_acc = [r["config"] for r in tuning_rows if r["best_val_acc"] == best["best_val_acc"]]

print("Ranking (validation only):")
for rank, r in enumerate(ranked, start=1):
    print(f"  {rank}. Config {r['config']}  val acc {r['best_val_acc']:.4f}  "
          f"val loss @ best epoch {r['val_loss_at_best_epoch']:.4f}")

print("\nBest configuration:")
print(f"  Config                   = {best['config']}")
print(f"  lr                       = {best['learning_rate']:g}")
print(f"  weight_decay             = {best['weight_decay']:g}")
print(f"  Best validation accuracy = {best['best_val_acc']:.4f} ({best['best_val_acc']:.2%})")
print(f"  Best epoch               = {best['best_epoch']}")
if len(tied_on_acc) > 1:
    print(f"  Note: configs {tied_on_acc} tied on validation accuracy; "
          f"the tie was broken by validation loss at the best epoch.")

BEST_CONFIG = {**best, "selection_rule": SELECTION_RULE, "tied_on_val_acc": tied_on_acc,
               "ranking": [r["config"] for r in ranked]}
with open(METRICS_DIR / "best_config.json", "w") as f:
    json.dump(BEST_CONFIG, f, indent=2)

with open(METRICS_DIR / "tuning_results.json", "w") as f:
    json.dump({"fixed": FIXED, "configs": CONFIGS, "selection_rule": SELECTION_RULE,
               "results": tuning_rows, "best_config": BEST_CONFIG}, f, indent=2)
print(f"\nSaved best_config.json and tuning_results.json to {METRICS_DIR}")

## 8. Final tuned model

The selected configuration's tuning run already followed the full Approach 3 protocol (train on train, keep the best-validation epoch), so **it does not need to be retrained**. Its best-validation checkpoint is reloaded into a fresh model. Reproducing the recorded validation accuracy confirms it is the selected model. It is then saved as the final tuned checkpoint.

In [ ]:
best_ckpt_path = RUN_DIR / best["checkpoint"]
final_model = create_resnet18_finetuned(num_classes=NUM_CLASSES)
final_checkpoint = load_checkpoint(best_ckpt_path, model=final_model, map_location=device)
final_model.to(device).eval()

assert final_checkpoint["epoch"] == best["best_epoch"], "checkpoint is not from the best epoch"
assert final_checkpoint["hyperparameters"]["learning_rate"] == best["learning_rate"]
assert final_checkpoint["hyperparameters"]["weight_decay"] == best["weight_decay"]

_, val_loader = make_train_val_loaders()
reloaded_val_loss, reloaded_val_acc = validate(final_model, val_loader, criterion, device)
print(f"Reloaded config {best['config']} checkpoint (epoch {final_checkpoint['epoch']})")
print(f"Validation acc: {reloaded_val_acc:.4f} (recorded best: {best['best_val_acc']:.4f})")
assert abs(reloaded_val_acc - best["best_val_acc"]) < 1e-4, "reloaded checkpoint does not reproduce val acc"

FINAL_CHECKPOINT_PATH = FINAL_DIR / "resnet18_finetuned_tuned_best.pt"
shutil.copy2(best_ckpt_path, FINAL_CHECKPOINT_PATH)
print(f"Final tuned checkpoint: {FINAL_CHECKPOINT_PATH}")

## 9. Test evaluation (run exactly once)

The configuration is now fixed, so the test set is loaded **for the first time** and the final tuned model is evaluated once. Nothing below changes the model or the choice of hyperparameters.

In [ ]:
set_seed(SEED)
_, _, test_loader = get_dataloaders(
    paths=paths, batch_size=FIXED["batch_size"], num_workers=FIXED["num_workers"], seed=SEED
)
assert len(test_loader.dataset) == EXPECTED_SIZES["test"]


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    y_true, y_pred, confidence = [], [], []
    for batch_images, batch_labels in loader:
        probs = torch.softmax(model(batch_images.to(device, non_blocking=True)), dim=1)
        conf, pred = probs.max(dim=1)
        y_true.append(batch_labels.numpy())
        y_pred.append(pred.cpu().numpy())
        confidence.append(conf.cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(confidence)


test_loss, _ = validate(final_model, test_loader, criterion, device)
y_true, y_pred, y_conf = predict(final_model, test_loader, device)
labels_idx = list(range(NUM_CLASSES))

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_idx, average="macro", zero_division=0
)
p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_idx, average="weighted", zero_division=0
)
TUNED_TEST_METRICS = {
    "test_loss": float(test_loss),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision_macro": float(p_macro),
    "recall_macro": float(r_macro),
    "f1_macro": float(f1_macro),
    "precision_weighted": float(p_weighted),
    "recall_weighted": float(r_weighted),
    "f1_weighted": float(f1_weighted),
    "num_test_samples": int(len(y_true)),
}
for k, v in TUNED_TEST_METRICS.items():
    print(f"{k:20s} {v:.4f}" if isinstance(v, float) else f"{k:20s} {v}")

report_text = classification_report(
    y_true, y_pred, labels=labels_idx, target_names=CLASS_NAMES, zero_division=0, digits=3
)
print("\n" + report_text)
pd.DataFrame(classification_report(
    y_true, y_pred, labels=labels_idx, target_names=CLASS_NAMES, zero_division=0, output_dict=True
)).T.to_csv(FINAL_DIR / "classification_report.csv")
(FINAL_DIR / "classification_report.txt").write_text(report_text)

test_df = test_loader.dataset.df
pd.DataFrame({
    "crop_path": test_df["crop_path"],
    "true_label": [CLASS_NAMES[i] for i in y_true],
    "pred_label": [CLASS_NAMES[i] for i in y_pred],
    "confidence": y_conf,
    "correct": y_true == y_pred,
}).to_csv(FINAL_DIR / "test_predictions.csv", index=False)
print(f"Misclassified test crops: {int((y_true != y_pred).sum())} / {len(y_true)}")

## 10. Confusion matrix (final tuned model)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(FINAL_DIR / "confusion_matrix.csv")
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums > 0)


def plot_confusion(matrix, title, filename, fmt):
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(labels_idx, labels=CLASS_NAMES, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(labels_idx, labels=CLASS_NAMES, fontsize=8)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    threshold = matrix.max() / 2 if matrix.max() > 0 else 0
    for i in labels_idx:
        for j in labels_idx:
            if matrix[i, j] > 0:
                ax.text(j, i, format(matrix[i, j], fmt), ha="center", va="center", fontsize=6,
                        color="white" if matrix[i, j] > threshold else "black")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=150)
    plt.show()
    plt.close(fig)


cfg_label = f"config {best['config']} (lr={best['learning_rate']:g}, wd={best['weight_decay']:g})"
plot_confusion(cm, f"Tuned ResNet-18, {cfg_label}: test confusion matrix (counts)",
               "final_confusion_matrix.png", "d")
plot_confusion(cm_norm, f"Tuned ResNet-18, {cfg_label}: test confusion matrix (row-normalised)",
               "final_confusion_matrix_normalized.png", ".2f")

## 11. Tuned model vs. Approach 3 baseline

The baseline values are the Approach 3 results (98.06% test accuracy). If the Approach 3 `results.json` is found on Drive, its values are used directly; otherwise the reported values below are used.

In [ ]:
BASELINE = {"source": "reported Approach 3 results", "accuracy": 0.9806, "f1_macro": 0.9680}
baseline_runs = sorted((OUTPUT_ROOT.parent / "resnet18_finetuned").glob("*/metrics/results.json"))
if baseline_runs:
    with open(baseline_runs[-1]) as f:
        baseline_test = json.load(f)["test"]
    BASELINE = {"source": baseline_runs[-1].as_posix(), **{
        k: baseline_test[k] for k in
        ("accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted") if k in baseline_test
    }}
print(f"Baseline source: {BASELINE['source']}")

rows = []
for key, label in [("accuracy", "Test accuracy"), ("precision_macro", "Macro precision"),
                   ("recall_macro", "Macro recall"), ("f1_macro", "Macro F1"), ("f1_weighted", "Weighted F1")]:
    base = BASELINE.get(key)
    tuned = TUNED_TEST_METRICS[key]
    rows.append({"Metric": label,
                 "Approach 3 baseline": f"{base:.2%}" if base is not None else "n/a",
                 "Tuned Approach 3": f"{tuned:.2%}",
                 "Change (pp)": f"{100 * (tuned - base):+.2f}" if base is not None else "n/a"})
comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(FINAL_DIR / "baseline_comparison.csv", index=False)
print(comparison_df.to_string(index=False))

delta_acc = TUNED_TEST_METRICS["accuracy"] - BASELINE["accuracy"]
verdict = "improved" if delta_acc > 0 else ("matched" if delta_acc == 0 else "decreased")
print(f"\nApproach 3 baseline: {BASELINE['accuracy']:.2%} test accuracy")
print(f"Tuned Approach 3:    {TUNED_TEST_METRICS['accuracy']:.2%} test accuracy")
print(f"Tuning {verdict} test accuracy by {100 * delta_acc:+.2f} percentage points.")

## 12. Final summary

The cell below writes the final summary from the actual results and saves `final_metrics.json`.

Talking points for the presentation:
- **Selection used validation only.** The test set was loaded once, after the choice, in Section 9.
- **Config A repeats the baseline settings.** It shows how reproducible the baseline is, and acts as the reference for B–D.
- **Trade-offs to look at:** validation accuracy vs. training time (smaller learning rates often need more epochs), and validation loss (confidence) vs. accuracy when runs tie. On a validation set of 454 images, one image is about 0.22 percentage points, so small differences are within noise.

In [ ]:
by_time = sorted(tuning_rows, key=lambda r: r["training_time_sec"])
summary_lines = [
    f"Best configuration: Config {best['config']} (lr = {best['learning_rate']:g}, weight decay = {best['weight_decay']:g}).",
    f"Why selected: highest best validation accuracy ({best['best_val_acc']:.2%})"
    + (f"; tied with {[c for c in tied_on_acc if c != best['config']]} and selected by lower validation loss "
       f"({best['val_loss_at_best_epoch']:.4f})" if len(tied_on_acc) > 1 else "")
    + ". The test set was not used for this choice.",
    f"Final test performance: accuracy {TUNED_TEST_METRICS['accuracy']:.2%}, macro precision "
    f"{TUNED_TEST_METRICS['precision_macro']:.2%}, macro recall {TUNED_TEST_METRICS['recall_macro']:.2%}, "
    f"macro F1 {TUNED_TEST_METRICS['f1_macro']:.2%}, weighted F1 {TUNED_TEST_METRICS['f1_weighted']:.2%}.",
    f"Compared with the Approach 3 baseline ({BASELINE['accuracy']:.2%}): tuning {verdict} test accuracy "
    f"by {100 * delta_acc:+.2f} percentage points.",
    "Trade-offs observed: "
    + "; ".join(
        f"Config {r['config']}: val acc {r['best_val_acc']:.2%}, best epoch {r['best_epoch']}, "
        f"{r['training_time_sec'] / 60:.1f} min" for r in tuning_rows
    )
    + f". Fastest run: Config {by_time[0]['config']}; slowest: Config {by_time[-1]['config']}.",
]
print("\n".join(f"- {line}" for line in summary_lines))

FINAL_METRICS = {
    "experiment": "approach3_resnet18_fully_finetuned_hyperparameter_tuning",
    "git_commit": GIT_COMMIT,
    "seed": SEED,
    "fixed_hyperparameters": FIXED,
    "tuned_hyperparameters": {"learning_rate": [c["learning_rate"] for c in CONFIGS],
                              "weight_decay": [c["weight_decay"] for c in CONFIGS]},
    "selection_rule": SELECTION_RULE,
    "best_config": BEST_CONFIG,
    "tuning_results": tuning_rows,
    "final_test_metrics": TUNED_TEST_METRICS,
    "baseline": BASELINE,
    "test_accuracy_change_pp": 100 * delta_acc,
    "model": {"total_parameters": TOTAL_PARAMS, "trainable_parameters": TRAINABLE_PARAMS,
              "pretrained_weights": FIXED["pretrained_weights"]},
    "hardware": HARDWARE,
    "summary": summary_lines,
    "artifacts": {
        "tuning_results_csv": "metrics/tuning_results.csv",
        "tuning_results_json": "metrics/tuning_results.json",
        "best_config": "metrics/best_config.json",
        "final_checkpoint": FINAL_CHECKPOINT_PATH.relative_to(RUN_DIR).as_posix(),
        "classification_report": "final_tuned_model/classification_report.csv",
        "confusion_matrix": "final_tuned_model/confusion_matrix.csv",
        "test_predictions": "final_tuned_model/test_predictions.csv",
        "baseline_comparison": "final_tuned_model/baseline_comparison.csv",
        "figures": sorted(p.relative_to(RUN_DIR).as_posix() for p in FIGURE_DIR.glob("*.png")),
    },
}
with open(METRICS_DIR / "final_metrics.json", "w") as f:
    json.dump(FINAL_METRICS, f, indent=2)

print(f"\nAll outputs saved under: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR).as_posix()}  ({p.stat().st_size / 1024:.0f} KB)")